# 06 · Feedback Checks

Checks for the four notes from the midpoint presentation, all on the final model (**logistic regression with change
features**) and the same split as `03`–`05`:

1. Should data be collected weekly?
2. Check age as a proxy, for SHAP
3. The system should agree with the actual rate at which older adults need help
4. Removing missing data: does it make a difference?

Show aggregate results only. Clear outputs before committing.

In [ ]:
import sys
from pathlib import Path

for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (root / "src" / "willow").exists():
        sys.path.insert(0, str(root / "src"))
        break
for name in list(sys.modules):
    if name == "willow" or name.startswith("willow."):
        del sys.modules[name]

import os
import numpy as np
import pandas as pd

from willow import models as md, metrics as mt
from willow.data import ROOT, load_panel
from willow.features import analytic_windows, build_windows, make_X

pd.set_option("display.max_columns", 40, "display.width", 160, "display.precision", 3)

OUTCOME = "y_any"
SPLIT_ROUND = 11          # same split as 03-05
FEATURES = "change"       # the final model
AGE_BANDS = ([0, 74, 84, 200], ["65-74", "75-84", "85+"])

a = analytic_windows(build_windows(load_panel()), OUTCOME)
train, test = md.person_time_split(a, SPLIT_ROUND)
y, people, w = test[OUTCOME].values, test.person_id.values, test["weight"].values
band = pd.cut(test.age, *AGE_BANDS[:1], labels=AGE_BANDS[1])

final = md.fit_logistic_model(train, test, FEATURES, OUTCOME)
print(f"final model: {final['n_features']} features, test AUROC {mt.core_metrics(y, final['p_test'], 0.5)['auroc']:.3f}")

## 1. Should Data Be Collected Weekly?

NHATS interviews people once a year, so the model has only learned from **changes about a year apart**. The data
can't tell us whether weekly changes carry the same signal. What we can say:

- Weekly answers will be noisier (good days and bad days) and ask more of older adults and caregivers.
- To match what the model learned, the app should compare each check-in with **about a year ago**, using the weekly
  answers as a running record rather than as week-to-week changes.
- Alerts should need a change that **lasts several check-ins**, so one bad week doesn't trigger "Talk to someone".
- The usability study is the place to test cadence: ask testers whether weekly or monthly feels right.

We keep "yearly data, weekly app" as a stated limitation.

## 2. Age As A Proxy, For SHAP

Public NHATS gives age only in **5-year groups**, stored here as group midpoints (67, 72 … 92, where 92 means 90+). So
"age" is already a coarse proxy. The checks below ask whether the model needs it, and why SHAP might rank it.

In [ ]:
print("age values used by the model (% of windows):")
display((a.age.value_counts(normalize=True).sort_index() * 100).round(1).rename("% of windows").to_frame().T)

no_age = md.fit_logistic_model(train, test, FEATURES, OUTCOME, drop=["age"])
age_drop = mt.paired_difference(y, final["p_test"], no_age["p_test"], people)
print("model without age minus model with age:")
age_drop

In [ ]:
# age overlaps with health measures, so SHAP splits credit between them
X = make_X(train, FEATURES)
related = ["cur_capacity_limits", "cur_dementia_class", "cur_walk_time", "cur_grip_max", "cur_self_rated_health", "n_areas_declined"]
X[["age"] + related].corr(method="spearman")["age"].drop("age").rename("correlation with age").round(2).to_frame()

In [ ]:
# where age ranks in each model's explanation (SHAP for trees from 04_trees, odds ratios for LR)
ranks = {}
for m in ("xgb", "rf"):
    f = os.path.join(ROOT, "results", f"model_{m}_change_shap.csv")
    if os.path.exists(f):
        s = pd.read_csv(f).reset_index()
        ranks[f"{m} SHAP"] = {r.feature: r["index"] + 1 for _, r in s.iterrows() if r.feature in ("age", "lives_alone")}
coefs = md.coefficients(final["model"], final["columns"]).reset_index()
ranks["LR odds ratio"] = {r.feature: r["index"] + 1 for _, r in coefs.iterrows() if r.feature in ("age", "lives_alone")}
pd.DataFrame(ranks).rename_axis("rank of")

## 3. Does The Model Agree With Actual Rates?

NHATS deliberately oversamples older people, so the raw sample has more events than the US population. A fair check
compares predicted and observed rates **for the same people**, then weights both to the US population with the NHATS
survey weights.

In [ ]:
rows = []
for b in AGE_BANDS[1]:
    pa = a[pd.cut(a.age, AGE_BANDS[0], labels=AGE_BANDS[1]) == b]
    m = (band == b).values
    rows.append({"age": b,
                 "US rate, all years (weighted %)": 100 * np.average(pa[OUTCOME], weights=pa.weight),
                 "test windows": int(m.sum()), "test events": int(y[m].sum()),
                 "test observed (weighted %)": 100 * np.average(y[m], weights=w[m]),
                 "model predicted (weighted %)": 100 * np.average(final["p_test"][m], weights=w[m]),
                 "observed / expected": mt.observed_expected(y[m], final["p_test"][m], w[m])})
rows.append({"age": "all", "US rate, all years (weighted %)": 100 * np.average(a[OUTCOME], weights=a.weight),
             "test windows": len(y), "test events": int(y.sum()),
             "test observed (weighted %)": 100 * np.average(y, weights=w),
             "model predicted (weighted %)": 100 * np.average(final["p_test"], weights=w),
             "observed / expected": mt.observed_expected(y, final["p_test"], w)})
rates = pd.DataFrame(rows).set_index("age")
rates

In [ ]:
# would recalibrating with survey weights change anything?
weighted = md.fit_logistic_model(train, test, FEATURES, OUTCOME, weight_col="weight")
pd.DataFrame({name: {"AUROC": mt.core_metrics(y, f["p_test"], 0.5)["auroc"],
                     "observed / expected (sample)": mt.observed_expected(y, f["p_test"]),
                     "observed / expected (US population)": mt.observed_expected(y, f["p_test"], w)}
              for name, f in {"current recalibration": final, "survey-weighted recalibration": weighted}.items()})

## 4. Removing Missing Data: Does It Make A Difference?

The current model fills gaps with the training median and adds a "was missing" flag. Three alternatives, each compared
on the same test people with a paired bootstrap:

In [ ]:
X_train, X_test = make_X(train, FEATURES), make_X(test, FEATURES)
missing = X_train.isna().mean().sort_values(ascending=False)
heavy = list(missing[missing > 0.10].index)
print(f"{len(heavy)} of {len(missing)} features are more than 10% missing in training data:")
display((100 * missing[heavy]).round(1).rename("% missing").to_frame().T)

complete_tr, complete_te = X_train.notna().all(axis=1).values, X_test.notna().all(axis=1).values
print(f"rows with nothing missing: train {complete_tr.mean():.0%}, test {complete_te.mean():.0%}")

In [ ]:
variants = {
    "Drop the heavily-missing features": md.fit_logistic_model(train, test, FEATURES, OUTCOME, drop=heavy),
    "Fill gaps without missing flags": md.fit_logistic_model(train, test, FEATURES, OUTCOME, missing_flags=False),
}
missing_checks = pd.concat({k: mt.paired_difference(y, final["p_test"], f["p_test"], people) for k, f in variants.items()})

# complete cases: train and test only on rows with nothing missing; compare on those test rows
cc = md.fit_logistic_model(train[complete_tr], test[complete_te], FEATURES, OUTCOME)
missing_checks = pd.concat([missing_checks, pd.concat({"Keep only rows with nothing missing": mt.paired_difference(
    y[complete_te], final["p_test"][complete_te], cc["p_test"], people[complete_te])})])
print("alternative minus current model:")
missing_checks

In [ ]:
# who would be dropped? people with gaps are sicker and older
pd.DataFrame({"nothing missing": {"test windows": int(complete_te.sum()), "new help (%)": 100 * y[complete_te].mean(),
                                  "aged 85+ (%)": 100 * (test.age[complete_te] >= 87).mean()},
              "something missing": {"test windows": int((~complete_te).sum()), "new help (%)": 100 * y[~complete_te].mean(),
                                    "aged 85+ (%)": 100 * (test.age[~complete_te] >= 87).mean()}}).T

## 5. Save Aggregate Outputs

In [ ]:
## Takeaways

**1. Weekly data.** NHATS can't test it, because the model has only seen yearly changes. The app should compare each
check-in with about a year ago and alert only on changes that last several check-ins. Ask about cadence in the
usability study, and keep it as a stated limitation.

**2. Age as a proxy.** Age is only known in 5-year groups, and the model doesn't need it: removing it changes AUROC by
+0.001 (95% CI −0.002 to +0.004). It overlaps with health measures (walking speed, grip, physical capacity,
dementia), so SHAP and odds ratios split credit between them. Report it as "age group" and don't present it as a
driver on its own. "Lives alone" ranks high (2nd in XGBoost SHAP) and *lowers* risk. That reflects receiving vs
needing help, not a protective effect.

**3. Agreement with actual rates.** Weighted to the US population, the model predicts 8.1% will newly need help vs
7.6% who actually do (observed/expected 0.94). By age group: 65–74 at 1.04, 75–84 at 0.83 (slightly over-predicted),
85+ at 1.08. Risk climbs with age as it does in the population (5.4% → 8.0% → 15.9%). Recalibrating with survey
weights changes almost nothing (0.94 → 0.96), because the model already accounts for age and health, so it isn't
needed.

**4. Missing data.** How gaps are handled doesn't change accuracy: dropping the 10 heavily-missing physical-test
features (−0.004), leaving out the missing flags (+0.002), or keeping only complete rows (−0.003) are all within noise.
But only 40–54% of rows are complete, and people with gaps are sicker (12.0% need help vs 5.5%) and older (25% vs 16%
aged 85+). Dropping them would make the data look even healthier, so filling gaps is the right choice. Dropping the
physical tests costing nothing is good news for the at-home app model.

## Takeaways

(filled in after the first run)